# M5: Optuna для LightGBM на 52 × 28

Подбираем параметры на полной сетке горизонтов: **52 даты отсчёта × 28 дней × 30 490 рядов = 44 393 440 обучающих строк**. Признаков 67; состав и порядок совпадают с ноутбуками 05 и 06. EWMA и короткие лаги не включены.

Бюджет — **до 40 попыток и 11 часов на один запуск поиска**, включая сборку и бинирование train. Первые две попытки используют параметры прежних trial 9 и trial 5, остальные предлагает Optuna. Отдельного длительного обучения контроля нет.

Окно подбора — **28 марта — 24 апреля 2016 года**. Число деревьев выбирается по WRMSSE: максимум 1600, early stopping 200. Это результат разработки на историческом периоде, а не официальный Kaggle score.

## С чем сравниваем

| Сохранённый вариант | WRMSSE | Bias | Деревьев |
|---|---:|---:|---:|
| Trial 9 на 52 × 28 | 0.501959 | −0.29% | 197 |
| Ансамбль моделей: 70% Trial 9 на 28 + 30% Trial 5 на 8 | 0.500879 | −0.52% | — |

Оба числа относятся к тому же историческому окну. Первая новая попытка заново проверит параметры trial 9 и сама выберет число деревьев. Её результат покажет, воспроизвёлся ли одиночный контроль в текущей среде. Номера попыток новой Optuna начинаются с нуля: **новый trial 0 — параметры прежнего trial 9**, новый trial 1 — прежнего trial 5.

В этом ноутбуке подбираем одиночную модель. Для улучшения ансамбля моделей позже понадобятся сохранённые прогнозы новых финалистов и прежних компонентов.

## 1. Запуск на Kaggle

1. Импортировать этот `.ipynb` в **новый** Kaggle Notebook.
2. Подключить **M5 Forecasting — Accuracy** через Input.
3. Выбрать **CPU / Accelerator → None**. Internet нужен для установки закреплённых версий библиотек.
4. Оставить настройки раздела 2 и выбрать **Save Version → Save & Run All**.

Для первого запуска старые ноутбуки и архивы не нужны: данные, 67 признаков и параметры двух стартовых кандидатов находятся здесь. Не требуется повторно запускать ноутбук 06.

После каждой завершённой попытки обновляется **`demand-planner_optuna_52x28_checkpoint.zip`**. В нём SQLite, параметры, таблицы, кривые обучения и модели с прогнозами двух лучших попыток. Архив забираем из Output успешно сохранённой версии.

Для продолжения подключить checkpoint **этого поиска 52 × 28** как Input. ZIP или распакованная история обнаруживаются автоматически; при нескольких копиях заполнить `RESUME_FROM`. Checkpoint прежней Optuna 52 × 8 к новой истории не подключается.

In [ ]:
from pathlib import Path
import importlib.metadata
import os
import subprocess
import sys

if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Этот ноутбук предназначен для Kaggle CPU. Локальный поиск не запускаем.")

PINNED_PACKAGES = {"lightgbm": "4.6.0", "optuna": "4.5.0"}
packages_to_install = []
for package, expected in PINNED_PACKAGES.items():
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        if package in sys.modules:
            raise RuntimeError(f"{package} уже импортирован в другой версии; перезапусти kernel.")
        packages_to_install.append(f"{package}=={expected}")
if packages_to_install:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--only-binary=:all:",
            *packages_to_install,
        ],
        check=True,
    )

os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"

from contextlib import redirect_stdout
from datetime import datetime, timezone
from functools import lru_cache
from time import perf_counter
from zipfile import ZIP_DEFLATED, ZipFile, is_zipfile
import fcntl
import gc
import gzip
import hashlib
import io
import json
import platform
import shutil
import sqlite3
import tempfile

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import scipy
from IPython.display import FileLink, display

pd.set_option("compute.use_bottleneck", False)
pd.set_option("display.max_columns", 30)
optuna.logging.set_verbosity(optuna.logging.INFO)
print("LightGBM:", lgb.__version__, "| Optuna:", optuna.__version__)
print("Bottleneck:", pd.get_option("compute.use_bottleneck"))

## 2. Настройки поиска

`TOTAL_TRIALS = 40` — общий лимит попыток в одной истории, включая две стартовые, не 40 дополнительных при каждом повторном запуске. COMPLETE, FAIL и PRUNED занимают попытку; WAITING ещё не запускалась.

`TIMEOUT_HOURS = 11` — бюджет одного вызова поиска. После этого callback завершит текущую попытку со статусом PRUNED; ранее завершённые сохранятся. Проверка времени выполняется между итерациями: возможен выход за срок на одну итерацию и сохранение файлов. Подготовку Dataset нельзя прервать этим callback. При продолжении из checkpoint начинается новый бюджет времени, общий лимит попыток остаётся прежним.

Сохранены 1600 деревьев, early stopping 200, Tweedie и WRMSSE на всех 12 уровнях. Поиск последовательный, четыре потока на модель. Отсечения по промежуточному качеству нет; PRUNED здесь означает только остановку по времени.

В прежнем поиске победитель подошёл к верхней границе learning rate и нижней границе степени Tweedie. Теперь диапазоны **0.025–0.15** и **1.01–1.90**. Остальные диапазоны прежние. Dataset бинируется один раз: `max_bin=255`, `feature_pre_filter=False`.

In [ ]:
SEED = 42
HORIZON = 28
N_ORIGINS = 52
HORIZONS_PER_ORIGIN = 28
NUM_THREADS = 4
TOTAL_TRIALS = 40
TIMEOUT_HOURS = 11
MAX_BOOST_ROUNDS = 1600
EARLY_STOPPING_ROUNDS = 200
RUN_SEARCH = True
RESUME_FROM = None

SEARCH_ORIGIN = pd.Timestamp("2016-03-27")
PUBLIC_ORIGIN = pd.Timestamp("2016-04-24")
cb_public_origin = PUBLIC_ORIGIN
STUDY_NAME = f"optuna_lgbm_67_cpu_52x{HORIZONS_PER_ORIGIN}_v1"
PROJECT_ROOT = Path("/kaggle/working/demand-planner")
RESULTS = PROJECT_ROOT / "results/m5_wrmsse"
MODELS = PROJECT_ROOT / "models"
RESULT_DIR = RESULTS / STUDY_NAME
MODEL_DIR = MODELS / STUDY_NAME
MANIFEST_PATH = RESULT_DIR / "study_manifest.json"
DATABASE_PATH = MODEL_DIR / "study.sqlite3"
CHECKPOINT_PATH = PROJECT_ROOT.parent / "demand-planner_optuna_52x28_checkpoint.zip"

REQUIRED_FILES = ("calendar.csv", "sales_train_validation.csv", "sell_prices.csv")
candidates = sorted(
    {
        path.parent
        for path in Path("/kaggle/input").rglob("calendar.csv")
        if all((path.parent / filename).is_file() for filename in REQUIRED_FILES)
    }
)
if len(candidates) != 1:
    raise FileNotFoundError(
        f"Подключи один набор M5 через Input. Подходящих папок: {len(candidates)}."
    )
DATA_RAW = candidates[0]
if HORIZONS_PER_ORIGIN != 28 or TOTAL_TRIALS < 1 or TIMEOUT_HOURS <= 0:
    raise ValueError("Некорректная плотность или бюджет поиска.")

FIXED_PARAMS = {
    "objective": "tweedie",
    "metric": "None",
    "boosting_type": "gbdt",
    "device_type": "cpu",
    "num_threads": NUM_THREADS,
    "seed": SEED,
    "deterministic": True,
    "force_col_wise": True,
    "verbosity": -1,
    "max_depth": -1,
    "max_bin": 255,
    "bin_construct_sample_cnt": 200000,
    "feature_pre_filter": False,
    "bagging_freq": 1,
    "min_sum_hessian_in_leaf": 0.001,
    "min_gain_to_split": 0.0,
}
SEED_TRIALS = [
    {
        "role": "seed_old_trial_009",
        "params": {
            "learning_rate": 0.09975125819860657,
            "num_leaves": 63,
            "min_data_in_leaf": 250,
            "lambda_l1": 0.8827248013286972,
            "lambda_l2": 1.740351379791848,
            "feature_fraction": 0.8563973660909573,
            "bagging_fraction": 0.7579657376635576,
            "tweedie_variance_power": 1.1026305531889093,
        },
    },
    {
        "role": "seed_old_trial_005",
        "params": {
            "learning_rate": 0.09579469687333796,
            "num_leaves": 63,
            "min_data_in_leaf": 250,
            "lambda_l1": 1.0682736700049809,
            "lambda_l2": 0.11322701075434688,
            "feature_fraction": 0.997966521134667,
            "bagging_fraction": 0.7128285798547571,
            "tweedie_variance_power": 1.1168250742845671,
        },
    },
]
REFERENCE_RESULTS = {
    "single_trial9_52x28": {
        "WRMSSE": 0.5019588456962026,
        "Bias": -0.002901704680924322,
        "trees": 197,
        "source": "notebook 06, " "density28_2016-03-27_52x28_trial_009/protocol.json",
    },
    "ensemble_70_30": {
        "WRMSSE": 0.500879332781371,
        "Bias": -0.005162882802935975,
        "source": "notebook 06, ensemble_pairs_v1; 70% trial9_28 + 30% trial5_8",
    },
}

SEARCH_SPACE = {
    "learning_rate": {"kind": "float", "low": 0.025, "high": 0.15, "log": True},
    "num_leaves": {"kind": "categorical", "choices": [31, 63, 127]},
    "min_data_in_leaf": {"kind": "categorical", "choices": [50, 100, 250, 500]},
    "lambda_l1": {"kind": "float", "low": 0.0, "high": 3.0},
    "lambda_l2": {"kind": "float", "low": 0.1, "high": 20.0, "log": True},
    "feature_fraction": {"kind": "float", "low": 0.75, "high": 1.0},
    "bagging_fraction": {"kind": "float", "low": 0.7, "high": 1.0},
    "tweedie_variance_power": {"kind": "float", "low": 1.01, "high": 1.9},
}

BASE_FEATURE_COLUMNS = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "mean_7d",
    "mean_28d",
    "mean_56d",
    "std_28d",
    "zero_share_28d",
    "trend_7d_28d",
    "weekday_mean_4w",
    "lag_28d",
    "lag_35d",
    "lag_42d",
    "lag_56d",
    "lag_364d",
    "horizon",
    "day_of_week",
    "day_of_month",
    "month",
    "week_of_year",
    "year",
    "snap",
    "has_event",
    "price_last_known",
    "price_observed_this_week",
    "price_change_1w",
    "price_change_4w",
    "price_vs_mean_4w",
    "price_vs_mean_13w",
    "price_vs_max_13w",
    "price_relative_std_13w",
    "price_observed_share_13w",
    "days_since_last_sale",
    "has_never_sold",
    "sale_day_share_7d",
    "sale_day_share_56d",
    "sale_frequency_change_7d_vs_prev21d",
    "mean_positive_sale_28d",
    "mean_positive_sale_56d",
    "positive_sale_cv2_56d",
    "max_daily_sale_28d",
    "store_dept_mean_7d",
    "store_dept_mean_28d",
    "store_dept_mean_56d",
    "store_dept_trend_7d_28d",
    "store_dept_trend_28d_56d",
    "store_dept_cv_28d",
    "store_dept_weekday_mean_4w",
    "store_dept_weekday_factor_4w",
    "item_share_store_dept_28d",
    "item_share_change_7d_28d",
    "other_stores_mean_7d",
    "other_stores_mean_28d",
    "other_stores_mean_56d",
    "other_stores_trend_7d_28d",
    "other_stores_trend_28d_56d",
    "other_stores_cv_28d",
    "other_stores_weekday_mean_4w",
    "other_stores_weekday_factor_4w",
    "own_vs_other_stores_mean_28d",
    "own_vs_other_stores_trend",
]

TARGET_PRICE_FEATURES = [
    "price_target_week",
    "price_target_vs_last",
    "price_target_vs_mean_13w",
    "price_target_missing",
]

FEATURE_COLUMNS = [*BASE_FEATURE_COLUMNS, *TARGET_PRICE_FEATURES]

CATEGORICAL_FEATURES = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

FEATURE_SOURCE_SHA256 = "7f870b6044f91821bf0a28613ba30cadb0503c25f0f07f1b87aa439a4cbae650"

if len(FEATURE_COLUMNS) != 67:
    raise ValueError("Ожидаются 67 принятых признаков.")
print("Данные:", DATA_RAW)
print("История поиска:", STUDY_NAME)
display(pd.DataFrame(SEARCH_SPACE).T)

## 3. История продаж и WRMSSE

Читаем 30 490 рядов из `sales_train_validation.csv`. Факты контрольного окна используем только для оценки; обучающие цели заканчиваются 27 марта. `sales_train_evaluation.csv` и факты зарезервированного июня этот поиск не читает.

Расчёт WRMSSE взят из ноутбука 04. Для этого поиска масштабы и денежные веса заканчиваются на дате отсчёта 27 марта. От них же строятся идентификаторы групп, используемые признаками отдела и других магазинов.

In [ ]:
calendar = pd.read_csv(DATA_RAW / "calendar.csv", parse_dates=["date"])
if calendar["date"].duplicated().any() or calendar["d"].duplicated().any():
    raise ValueError("В календаре есть повторяющиеся дни.")

In [ ]:
identifier_columns = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]

series_metadata = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv",
    usecols=identifier_columns,
    dtype="string",
)

assert not series_metadata.duplicated(["item_id", "store_id"]).any()
assert len(series_metadata) == 30_490
assert series_metadata["store_id"].nunique() == 10

M5_LEVELS = {
    "Total": [],
    "State": ["state_id"],
    "Store": ["store_id"],
    "Category": ["cat_id"],
    "Department": ["dept_id"],
    "State / Category": ["state_id", "cat_id"],
    "State / Department": ["state_id", "dept_id"],
    "Store / Category": ["store_id", "cat_id"],
    "Store / Department": ["store_id", "dept_id"],
    "Item": ["item_id"],
    "Item / State": ["item_id", "state_id"],
    "Item / Store": ["item_id", "store_id"],
}

hierarchy = pd.DataFrame(
    [
        {
            "level": level,
            "series": (series_metadata.groupby(columns, observed=True).ngroups if columns else 1),
        }
        for level, columns in M5_LEVELS.items()
    ]
).set_index("level")

assert int(hierarchy["series"].sum()) == 42_840
display(hierarchy)
print(f"Всего оцениваемых рядов: {hierarchy['series'].sum():,}")

In [ ]:
M5_KEYS = ["store_id", "item_id"]
TRAIN_END_DAY = 1913

train_day_columns = [f"d_{day}" for day in range(1, TRAIN_END_DAY + 1)]

series_index = pd.MultiIndex.from_frame(series_metadata[M5_KEYS])

train_daily = pd.read_csv(
    DATA_RAW / "sales_train_validation.csv",
    usecols=[*M5_KEYS, *train_day_columns],
    dtype={
        "store_id": "string",
        "item_id": "string",
        **{column: "int32" for column in train_day_columns},
    },
)

train_daily = train_daily.set_index(M5_KEYS).reindex(series_index).loc[:, train_day_columns]

train_daily.columns = pd.DatetimeIndex(
    calendar.set_index("d").loc[train_day_columns, "date"],
    name="date",
)

weight_dates = train_daily.loc[:, :SEARCH_ORIGIN].columns[-28:]

weight_weeks = calendar.set_index("date").loc[weight_dates, "wm_yr_wk"].unique()

price_parts = []

for chunk in pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    dtype={
        "store_id": "string",
        "item_id": "string",
        "wm_yr_wk": "int32",
        "sell_price": "float64",
    },
    chunksize=200_000,
):
    selected = chunk.loc[chunk["wm_yr_wk"].isin(weight_weeks)]

    if not selected.empty:
        price_parts.append(selected.copy())

prices_for_weights = pd.concat(
    price_parts,
    ignore_index=True,
)

del price_parts

print(f"Рядов: {len(train_daily):,}")
print(f"Дней истории: {train_daily.shape[1]:,}")
print(
    "Период для весов:",
    weight_dates.min().date(),
    "—",
    weight_dates.max().date(),
)

In [ ]:
from scipy.sparse import csr_matrix


def calculate_rmsse_scale(history_values, aggregation, block_size=128):
    n_groups = aggregation.shape[0]
    n_days = history_values.shape[1]

    first_nonzero = np.full(n_groups, n_days, dtype=np.int32)
    squared_difference_sum = np.zeros(n_groups, dtype=np.float64)
    previous = np.zeros(n_groups, dtype=np.float64)

    for start in range(0, n_days, block_size):
        stop = min(start + block_size, n_days)

        block = np.asarray(
            aggregation @ history_values[:, start:stop],
            dtype=np.float64,
        )

        positive = block > 0
        newly_active = (first_nonzero == n_days) & positive.any(axis=1)

        if newly_active.any():
            first_nonzero[newly_active] = start + positive[newly_active].argmax(axis=1)

        differences = np.empty_like(block)
        differences[:, 0] = block[:, 0] - previous
        differences[:, 1:] = np.diff(block, axis=1)

        positions = np.arange(start, stop)
        exclude = positions[None, :] <= first_nonzero[:, None]
        differences[exclude] = 0.0

        squared_difference_sum += np.einsum(
            "ij,ij->i",
            differences,
            differences,
        )

        previous = block[:, -1].copy()

    difference_count = n_days - first_nonzero - 1

    return np.divide(
        squared_difference_sum,
        difference_count,
        out=np.zeros(n_groups, dtype=np.float64),
        where=difference_count > 0,
    )


class M5WRMSSE:
    def __init__(
        self,
        history,
        metadata,
        calendar,
        prices,
        levels,
        horizon=28,
    ):
        if len(metadata) != 30_490 or len(levels) != 12:
            raise ValueError("Для полного M5 нужны 30 490 рядов и 12 уровней.")

        self.series_index = pd.MultiIndex.from_frame(metadata[M5_KEYS])

        if (
            self.series_index.has_duplicates
            or history.index.has_duplicates
            or len(history) != len(self.series_index)
            or not self.series_index.isin(history.index).all()
        ):
            raise ValueError("Идентификаторы истории и metadata не совпадают.")

        history = history.reindex(self.series_index)

        if history.shape[1] < 28:
            raise ValueError("Для расчёта весов нужны минимум 28 дней.")

        if not isinstance(history.columns, pd.DatetimeIndex):
            raise TypeError("Столбцы истории должны содержать даты.")

        expected_history_dates = pd.date_range(
            history.columns[0],
            periods=history.shape[1],
            freq="D",
        )

        if not history.columns.equals(expected_history_dates):
            raise ValueError("В истории есть пропуски или нарушен порядок дней.")

        history_values = history.to_numpy(copy=False)

        if not np.isfinite(history_values).all() or (history_values < 0).any():
            raise ValueError("История должна содержать конечные неотрицательные продажи.")

        self.horizon = horizon
        self.n_levels = len(levels)
        self.forecast_dates = pd.date_range(
            history.columns[-1] + pd.Timedelta(days=1),
            periods=horizon,
            freq="D",
            name="date",
        )

        recent_dates = history.columns[-28:]

        week_codes = calendar.set_index("date").reindex(recent_dates)["wm_yr_wk"].to_numpy()

        if pd.isna(week_codes).any():
            raise ValueError("В календаре отсутствуют даты для расчёта весов.")

        known_prices = prices["sell_price"].dropna().to_numpy()

        if not np.isfinite(known_prices).all() or (known_prices <= 0).any():
            raise ValueError("Известные цены должны быть конечными и положительными.")

        price_table = prices.pivot(
            index=M5_KEYS,
            columns="wm_yr_wk",
            values="sell_price",
        )

        daily_prices = price_table.reindex(
            index=self.series_index,
            columns=week_codes,
        ).to_numpy(dtype=np.float64)

        recent_units = history.iloc[:, -28:].to_numpy(dtype=np.float64)

        missing_price_with_sales = (recent_units > 0) & ~np.isfinite(daily_prices)

        if missing_price_with_sales.any():
            raise ValueError(
                "Есть положительные продажи без цены: веса нельзя посчитать корректно."
            )

        dollar_sales = (recent_units * np.nan_to_num(daily_prices, nan=0.0)).sum(axis=1)

        total_dollar_sales = dollar_sales.sum()

        if total_dollar_sales <= 0:
            raise ValueError("Суммарные денежные продажи равны нулю.")

        metadata = metadata.reset_index(drop=True)

        self.level_specs = {}
        level_rows = []

        for level_name, columns in levels.items():
            if columns:
                group_index = pd.MultiIndex.from_frame(metadata[columns])
                codes, groups = pd.factorize(
                    group_index,
                    sort=True,
                )
                n_groups = len(groups)
            else:
                codes = np.zeros(len(metadata), dtype=np.int32)
                n_groups = 1

            aggregation = csr_matrix(
                (
                    np.ones(len(metadata), dtype=np.float64),
                    (codes, np.arange(len(metadata))),
                ),
                shape=(n_groups, len(metadata)),
            )

            print(f"Масштабы: {level_name} — {n_groups:,} рядов")

            scale = calculate_rmsse_scale(
                history_values,
                aggregation,
            )

            group_dollar_sales = np.asarray(aggregation @ dollar_sales).ravel()

            weights = group_dollar_sales / total_dollar_sales / self.n_levels

            if ((scale <= 0) & (weights > 0)).any():
                raise ValueError(
                    f"{level_name}: есть ряд с положительным весом " "и нулевым масштабом RMSSE."
                )

            self.level_specs[level_name] = {
                "aggregation": aggregation,
                "scale": scale,
                "weights": weights,
            }

            level_rows.append(
                {
                    "level": level_name,
                    "series": n_groups,
                    "weight_sum": weights.sum(),
                }
            )

        self.level_info = pd.DataFrame(level_rows).set_index("level")

        if self.level_info["series"].sum() != 42_840:
            raise ValueError("Получено неверное число рядов иерархии.")

        if not np.isclose(self.level_info["weight_sum"].sum(), 1.0):
            raise ValueError("Сумма весов WRMSSE отличается от единицы.")

    def _aligned_values(self, frame):
        if not isinstance(frame, pd.DataFrame):
            raise TypeError("Передай DataFrame с идентификаторами и датами.")

        if (
            frame.index.has_duplicates
            or len(frame) != len(self.series_index)
            or not self.series_index.isin(frame.index).all()
        ):
            raise ValueError("Набор товаров и магазинов не совпадает.")

        if not frame.columns.equals(self.forecast_dates):
            raise ValueError("Столбцы должны содержать следующие 28 дат " "в правильном порядке.")

        values = frame.reindex(self.series_index).to_numpy(dtype=np.float64)

        if not np.isfinite(values).all():
            raise ValueError("В фактах или прогнозах есть NaN либо бесконечность.")

        return values

    def score(self, actual, prediction):
        actual_values = self._aligned_values(actual)
        prediction_values = self._aligned_values(prediction)

        bottom_errors = prediction_values - actual_values
        rows = []

        for level_name, spec in self.level_specs.items():
            aggregated_errors = spec["aggregation"] @ bottom_errors

            mse = np.mean(
                np.square(aggregated_errors),
                axis=1,
            )

            rmsse = np.sqrt(
                np.divide(
                    mse,
                    spec["scale"],
                    out=np.zeros_like(mse),
                    where=spec["scale"] > 0,
                )
            )

            contribution = float(np.dot(spec["weights"], rmsse))

            rows.append(
                {
                    "level": level_name,
                    "weighted_RMSSE": contribution * self.n_levels,
                    "contribution": contribution,
                }
            )

        report = pd.DataFrame(rows).set_index("level")
        overall = float(report["contribution"].sum())

        return overall, report

In [ ]:
wrmsse_evaluator = M5WRMSSE(
    history=train_daily.loc[:, :SEARCH_ORIGIN],
    metadata=series_metadata,
    calendar=calendar,
    prices=prices_for_weights,
    levels=M5_LEVELS,
    horizon=HORIZON,
)

display(
    wrmsse_evaluator.level_info.style.format(
        {
            "series": "{:,.0f}",
            "weight_sum": "{:.6f}",
        }
    )
)

print(
    "Всего рядов:",
    f"{wrmsse_evaluator.level_info['series'].sum():,}",
)

print(
    "Сумма весов:",
    f"{wrmsse_evaluator.level_info['weight_sum'].sum():.6f}",
)

print(
    "Даты будущего прогноза:",
    wrmsse_evaluator.forecast_dates.min().date(),
    "—",
    wrmsse_evaluator.forecast_dates.max().date(),
)

In [ ]:
def price_evaluate(evaluator, actual, prediction):
    score, report = evaluator.score(actual, prediction)
    actual_total = (
        actual.reindex(index=prediction.index, columns=prediction.columns)
        .to_numpy(dtype=np.float64)
        .sum()
    )
    metrics = {
        "WRMSSE": float(score),
        "Bias": float(prediction.to_numpy().sum() / actual_total - 1),
    }
    return (metrics, report)

## 4. Принятые признаки

Состав и порядок 67 столбцов сохранены из ноутбука 04. Здесь нет автоматического отбора или добавления новых гипотез. Все признаки продаж ограничиваются датой отсчёта. Будущие цены допустимы в рамках предоставленного M5 ценового плана; в прикладной задаче для них нужен известный заранее план цен.

### Продажи и календарь

Средние, разброс, нули, изменение уровня, недельный профиль и лаги от 28 дней. Календарь и SNAP относятся к целевому дню.

In [ ]:
daily_categorical_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
]

daily_history = train_daily.reindex(wrmsse_evaluator.series_index)

daily_history_dates = daily_history.columns
daily_history_values = daily_history.to_numpy(copy=False)

daily_identifiers = (
    series_metadata.set_index(M5_KEYS)
    .loc[wrmsse_evaluator.series_index]
    .reset_index()
    .loc[:, daily_categorical_features]
)

daily_identifiers.index = wrmsse_evaluator.series_index

for column in daily_categorical_features:
    daily_identifiers[column] = daily_identifiers[column].astype("category")

daily_calendar = calendar.set_index("date")


def make_daily_features(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)

    horizon = (target_date - forecast_origin).days

    if not 1 <= horizon <= HORIZON:
        raise ValueError("Горизонт должен быть от 1 до 28 дней.")

    origin_position = daily_history_dates.get_indexer([forecast_origin])[0]

    if origin_position < 55:
        raise ValueError(
            "Дата прогноза должна присутствовать в истории " "и иметь минимум 56 доступных дней."
        )

    X = daily_identifiers.copy()
    known_history = daily_history_values[:, : origin_position + 1]

    for window in (7, 28, 56):
        X[f"mean_{window}d"] = known_history[:, -window:].mean(axis=1).astype("float32")

    recent_28 = known_history[:, -28:]

    X["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")

    X["zero_share_28d"] = (recent_28 == 0).mean(axis=1).astype("float32")

    X["trend_7d_28d"] = (X["mean_7d"] - X["mean_28d"]).astype("float32")

    recent_dates = daily_history_dates[origin_position - 27 : origin_position + 1]

    weekday_mask = recent_dates.dayofweek == target_date.dayofweek

    X["weekday_mean_4w"] = recent_28[:, weekday_mask].mean(axis=1).astype("float32")

    for lag in (28, 35, 42, 56, 364):
        source_date = target_date - pd.Timedelta(days=lag)

        if source_date > forecast_origin:
            raise ValueError("Лаг обращается к неизвестному будущему.")

        source_position = daily_history_dates.get_indexer([source_date])[0]

        if source_position >= 0:
            X[f"lag_{lag}d"] = daily_history_values[:, source_position].astype("float32")
        else:
            X[f"lag_{lag}d"] = np.float32(np.nan)

    calendar_values = daily_calendar.loc[target_date]

    target_features = {
        "horizon": horizon,
        "day_of_week": target_date.dayofweek,
        "day_of_month": target_date.day,
        "month": target_date.month,
        "week_of_year": int(target_date.isocalendar().week),
        "year": target_date.year,
    }

    for name, value in target_features.items():
        X[name] = np.int16(value)

    snap_by_state = {state: int(calendar_values[f"snap_{state}"]) for state in ("CA", "TX", "WI")}

    X["snap"] = X["state_id"].map(snap_by_state).astype("int8")

    X["has_event"] = np.int8(
        pd.notna(calendar_values["event_name_1"]) or pd.notna(calendar_values["event_name_2"])
    )

    return X

### История цены

Последняя известная цена, исторические изменения, относительный уровень и разброс. Bottleneck отключён до подготовки.

In [ ]:
price_data = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int32",
        "sell_price": "float64",
    },
)
price_week_starts = calendar.groupby("wm_yr_wk")["date"].min().sort_values()
price_week_order = price_week_starts.loc[price_week_starts <= cb_public_origin].index
price_data = price_data.loc[price_data["wm_yr_wk"].isin(price_week_order)].copy()
price_week_table = (
    price_data.pivot(index=M5_KEYS, columns="wm_yr_wk", values="sell_price")
    .reindex(index=wrmsse_evaluator.series_index, columns=price_week_order)
    .astype("float32")
)
price_last_known_table = price_week_table.ffill(axis=1)


@lru_cache(maxsize=64)
def price_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    week = int(daily_calendar.loc[forecast_origin, "wm_yr_wk"])
    position = price_week_table.columns.get_loc(week)
    if position < 12:
        raise ValueError("Для признаков цены нужны 13 недель истории.")
    recent_4 = price_week_table.iloc[:, position - 3 : position + 1]
    recent_13 = price_week_table.iloc[:, position - 12 : position + 1]
    current = price_last_known_table.iloc[:, position]
    previous_1 = price_last_known_table.iloc[:, position - 1]
    previous_4 = price_last_known_table.iloc[:, position - 4]
    mean_4 = recent_4.mean(axis=1)
    mean_13 = recent_13.mean(axis=1)
    return pd.DataFrame(
        {
            "price_last_known": current,
            "price_observed_this_week": price_week_table.iloc[:, position].notna(),
            "price_change_1w": current / previous_1 - 1,
            "price_change_4w": current / previous_4 - 1,
            "price_vs_mean_4w": current / mean_4 - 1,
            "price_vs_mean_13w": current / mean_13 - 1,
            "price_vs_max_13w": current / recent_13.max(axis=1) - 1,
            "price_relative_std_13w": recent_13.std(axis=1, ddof=0) / mean_13,
            "price_observed_share_13w": recent_13.notna().mean(axis=1),
        }
    ).astype("float32")


def make_daily_features_with_price(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    base = make_daily_features(forecast_origin=forecast_origin, target_date=target_date)
    return base.join(price_features_at_origin(forecast_origin), validate="one_to_one")

### Регулярность продаж

Давность последней продажи, частота ненулевых дней, размер положительной продажи и его изменчивость.

In [ ]:
@lru_cache(maxsize=64)
def regularity_features_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Дата должна присутствовать в истории и иметь минимум 56 доступных дней.")
    known = daily_history_values[:, : position + 1]
    positive_history = known > 0
    has_ever_sold = positive_history.any(axis=1)
    days_since_sale = positive_history[:, ::-1].argmax(axis=1).astype("float32")
    days_since_sale[~has_ever_sold] = np.nan
    recent_56 = known[:, -56:].astype("float64")
    recent_28 = recent_56[:, -28:]
    positive_56 = positive_history[:, -56:]
    positive_28 = positive_56[:, -28:]
    count_56 = positive_56.sum(axis=1)
    count_28 = positive_28.sum(axis=1)
    sum_56 = recent_56.sum(axis=1)
    sum_28 = recent_28.sum(axis=1)
    mean_positive_56 = np.divide(
        sum_56, count_56, out=np.full(len(known), np.nan), where=count_56 > 0
    )
    mean_positive_28 = np.divide(
        sum_28, count_28, out=np.full(len(known), np.nan), where=count_28 > 0
    )
    squared_sum_56 = np.einsum("ij,ij->i", recent_56, recent_56)
    correction = np.divide(
        np.square(sum_56), count_56, out=np.zeros(len(known)), where=count_56 > 0
    )
    positive_variance_56 = np.divide(
        np.maximum(squared_sum_56 - correction, 0.0),
        count_56 - 1,
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    positive_cv2_56 = np.divide(
        positive_variance_56,
        np.square(mean_positive_56),
        out=np.full(len(known), np.nan),
        where=count_56 > 1,
    )
    frequency_7 = positive_56[:, -7:].mean(axis=1)
    frequency_previous_21 = positive_56[:, -28:-7].mean(axis=1)
    return pd.DataFrame(
        {
            "days_since_last_sale": days_since_sale,
            "has_never_sold": ~has_ever_sold,
            "sale_day_share_7d": frequency_7,
            "sale_day_share_56d": positive_56.mean(axis=1),
            "sale_frequency_change_7d_vs_prev21d": frequency_7 - frequency_previous_21,
            "mean_positive_sale_28d": mean_positive_28,
            "mean_positive_sale_56d": mean_positive_56,
            "positive_sale_cv2_56d": positive_cv2_56,
            "max_daily_sale_28d": recent_28.max(axis=1),
        },
        index=wrmsse_evaluator.series_index,
    ).astype("float32")


def make_daily_features_with_regularity(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    X = make_daily_features_with_price(forecast_origin, target_date)
    return X.join(regularity_features_at_origin(forecast_origin), validate="one_to_one")

### Отдел в магазине

История суммарных продаж отдела и положение товара относительно своей группы.

In [ ]:
group_aggregation = wrmsse_evaluator.level_specs["Store / Department"]["aggregation"]
group_membership = group_aggregation.tocoo()
group_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
group_codes[group_membership.col] = group_membership.row
group_daily_sales = np.asarray(group_aggregation @ daily_history_values, dtype=np.float64)


def group_safe_ratio(numerator, denominator):
    numerator = np.asarray(numerator, dtype=np.float64)
    denominator = np.asarray(denominator, dtype=np.float64)
    return np.divide(
        numerator,
        denominator,
        out=np.full(numerator.shape, np.nan),
        where=denominator > 0,
    )


@lru_cache(maxsize=64)
def group_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для групповых признаков нужны 56 доступных дней.")
    known = group_daily_sales[:, : position + 1]
    recent_28 = known[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {f"mean_{window}d": known[:, -window:].mean(axis=1) for window in (7, 28, 56)}
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1)
    statistics["weekday_mean_4w"] = np.column_stack(
        [recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1) for weekday in range(7)]
    )
    return statistics


def make_daily_features_with_groups(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_regularity(forecast_origin, target_date)
    statistics = group_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"][group_codes]
    mean_28 = statistics["mean_28d"][group_codes]
    mean_56 = statistics["mean_56d"][group_codes]
    weekday_mean = statistics["weekday_mean_4w"][group_codes, target_date.dayofweek]
    item_share_7 = group_safe_ratio(X["mean_7d"].to_numpy(), mean_7)
    item_share_28 = group_safe_ratio(X["mean_28d"].to_numpy(), mean_28)
    added_features = {
        "store_dept_mean_7d": mean_7,
        "store_dept_mean_28d": mean_28,
        "store_dept_mean_56d": mean_56,
        "store_dept_trend_7d_28d": group_safe_ratio(mean_7, mean_28) - 1,
        "store_dept_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "store_dept_cv_28d": group_safe_ratio(statistics["std_28d"][group_codes], mean_28),
        "store_dept_weekday_mean_4w": weekday_mean,
        "store_dept_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "item_share_store_dept_28d": item_share_28,
        "item_share_change_7d_28d": item_share_7 - item_share_28,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X

### Тот же товар в других магазинах

Агрегаты остальных магазинов исключают собственный магазин товара.

In [ ]:
cross_store_aggregation = wrmsse_evaluator.level_specs["Item"]["aggregation"]
cross_store_membership = cross_store_aggregation.tocoo()
cross_store_codes = np.empty(len(wrmsse_evaluator.series_index), dtype=np.int32)
cross_store_codes[cross_store_membership.col] = cross_store_membership.row
cross_store_counts = np.asarray(cross_store_aggregation.sum(axis=1)).ravel()
cross_store_other_counts = cross_store_counts[cross_store_codes] - 1
cross_store_daily_totals = np.asarray(
    cross_store_aggregation @ daily_history_values, dtype=np.float64
)


@lru_cache(maxsize=64)
def cross_store_statistics_at_origin(forecast_origin):
    forecast_origin = pd.Timestamp(forecast_origin)
    position = daily_history_dates.get_indexer([forecast_origin])[0]
    if position < 55:
        raise ValueError("Для признаков других магазинов нужны 56 дней истории.")
    start = position - 55
    network_sales = cross_store_daily_totals[cross_store_codes, start : position + 1]
    own_sales = daily_history_values[:, start : position + 1]
    other_sales = (network_sales - own_sales) / cross_store_other_counts[:, None]
    recent_28 = other_sales[:, -28:]
    recent_dates = daily_history_dates[position - 27 : position + 1]
    statistics = {
        f"mean_{window}d": other_sales[:, -window:].mean(axis=1).astype("float32")
        for window in (7, 28, 56)
    }
    statistics["std_28d"] = recent_28.std(axis=1, ddof=1).astype("float32")
    statistics["weekday_mean_4w"] = np.column_stack(
        [recent_28[:, recent_dates.dayofweek == weekday].mean(axis=1) for weekday in range(7)]
    ).astype("float32")
    return statistics


def make_daily_features_with_cross_store(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    X = make_daily_features_with_groups(forecast_origin, target_date)
    statistics = cross_store_statistics_at_origin(forecast_origin)
    mean_7 = statistics["mean_7d"]
    mean_28 = statistics["mean_28d"]
    mean_56 = statistics["mean_56d"]
    weekday_mean = statistics["weekday_mean_4w"][:, target_date.dayofweek]
    own_trend = group_safe_ratio(X["mean_7d"].to_numpy(), X["mean_28d"].to_numpy()) - 1
    other_trend = group_safe_ratio(mean_7, mean_28) - 1
    added_features = {
        "other_stores_mean_7d": mean_7,
        "other_stores_mean_28d": mean_28,
        "other_stores_mean_56d": mean_56,
        "other_stores_trend_7d_28d": other_trend,
        "other_stores_trend_28d_56d": group_safe_ratio(mean_28, mean_56) - 1,
        "other_stores_cv_28d": group_safe_ratio(statistics["std_28d"], mean_28),
        "other_stores_weekday_mean_4w": weekday_mean,
        "other_stores_weekday_factor_4w": group_safe_ratio(weekday_mean, mean_28),
        "own_vs_other_stores_mean_28d": group_safe_ratio(X["mean_28d"].to_numpy(), mean_28),
        "own_vs_other_stores_trend": own_trend - other_trend,
    }
    for column, values in added_features.items():
        X[column] = values.astype("float32")
    return X

### Цена целевой недели

Четыре принятых ценовых признака и единые категории для train и validation. EWMA и короткие лаги отсутствуют.

In [ ]:
CATEGORY_TYPES = {
    column: pd.CategoricalDtype(
        categories=daily_identifiers[column].cat.categories,
        ordered=False,
    )
    for column in CATEGORICAL_FEATURES
}

target_price_source = pd.read_csv(
    DATA_RAW / "sell_prices.csv",
    usecols=["store_id", "item_id", "wm_yr_wk", "sell_price"],
    dtype={
        "store_id": "category",
        "item_id": "category",
        "wm_yr_wk": "int32",
        "sell_price": "float32",
    },
)
observed_prices = target_price_source["sell_price"].dropna().to_numpy()
if not np.isfinite(observed_prices).all() or (observed_prices <= 0).any():
    raise ValueError("Некорректные наблюдаемые цены.")
target_week_prices = (
    target_price_source.pivot(index=M5_KEYS, columns="wm_yr_wk", values="sell_price")
    .reindex(index=wrmsse_evaluator.series_index)
    .astype("float32")
)
del target_price_source, observed_prices
gc.collect()


def prepare_categories(frame):
    for column, dtype in CATEGORY_TYPES.items():
        frame[column] = frame[column].astype(dtype)
        if frame[column].isna().any():
            raise ValueError(f"Неизвестная категория в {column}.")
    return frame


def relative_price_change(numerator, denominator):
    result = np.full(len(numerator), np.nan, dtype=np.float32)
    valid = np.isfinite(numerator) & np.isfinite(denominator) & (denominator > 0)
    np.divide(numerator, denominator, out=result, where=valid)
    return result - np.float32(1)


@lru_cache(maxsize=64)
def historical_price_mean_13w(forecast_origin):
    week = int(daily_calendar.loc[forecast_origin, "wm_yr_wk"])
    position = price_week_table.columns.get_loc(week)
    if position < 12:
        raise ValueError("Для среднего цены нужны 13 недель истории.")
    return (
        price_week_table.iloc[:, position - 12 : position + 1]
        .mean(axis=1)
        .to_numpy(dtype=np.float32)
    )


def make_daily_features_final(forecast_origin, target_date):
    forecast_origin = pd.Timestamp(forecast_origin)
    target_date = pd.Timestamp(target_date)
    frame = make_daily_features_with_cross_store(forecast_origin, target_date)
    if frame.columns.tolist() != BASE_FEATURE_COLUMNS or not frame.index.equals(
        wrmsse_evaluator.series_index
    ):
        raise ValueError("Изменились базовые признаки или порядок рядов.")
    target_week = int(daily_calendar.loc[target_date, "wm_yr_wk"])
    target_price = target_week_prices[target_week].to_numpy()
    last_price = frame["price_last_known"].to_numpy()
    frame["price_target_week"] = target_price
    frame["price_target_vs_last"] = relative_price_change(target_price, last_price)
    frame["price_target_vs_mean_13w"] = relative_price_change(
        target_price, historical_price_mean_13w(forecast_origin)
    )
    frame["price_target_missing"] = (~np.isfinite(target_price)).astype("float32")
    return prepare_categories(frame.loc[:, FEATURE_COLUMNS])

## 5. Полный план 52 × 28

Для каждой из 52 еженедельных дат отсчёта используем все горизонты от 1 до 28. Обучающие цели заканчиваются 27 марта; признаки продаж знают только историю до своей даты отсчёта. Внутри каждой пары «дата отсчёта — целевой день» идут все 30 490 рядов в одном порядке.

Сохраняем `training_plan.csv` и его хеш. Ожидаем 1456 пар дат, 44 393 440 строк train и 853 720 строк validation. Категории задаются одинаково для обоих Dataset. Бинирование выполняется один раз для всего поиска. Таблица признаков до бинирования занимала около 9.76 GiB; это не весь расход RAM процесса.

In [ ]:
def make_training_plan(label_cutoff, horizons_per_origin=16):
    if horizons_per_origin not in (2, 4, 8, 16, 28):
        raise ValueError("Допустимы планы 52 × 2, 4, 8, 16 или 28.")
    label_cutoff = pd.Timestamp(label_cutoff)
    origins = pd.date_range(
        end=label_cutoff - pd.Timedelta(days=HORIZON),
        periods=N_ORIGINS,
        freq="7D",
    )
    rows = []
    for number, origin in enumerate(origins):
        first = 5 * number % HORIZON + 1
        old = {first, (5 * number + HORIZON // 2) % HORIZON + 1}
        if number == N_ORIGINS - 1:
            old = {1, HORIZON}
        four = old | (
            {10, 19}
            if number == N_ORIGINS - 1
            else {(first - 1 + shift) % HORIZON + 1 for shift in (7, 21)}
        )
        eight = four | (
            {5, 14, 23, 26}
            if number == N_ORIGINS - 1
            else {(first - 1 + shift) % HORIZON + 1 for shift in (3, 10, 17, 24)}
        )
        if len(old) != 2 or len(four) != 4 or len(eight) != 8:
            raise ValueError("Изменился исходный план горизонтов.")
        horizons = set({2: old, 4: four}.get(horizons_per_origin, eight))
        while len(horizons) < horizons_per_origin:
            candidates = set(range(1, HORIZON + 1)) - horizons

            def separation(candidate):
                return min(
                    min(abs(candidate - old), HORIZON - abs(candidate - old)) for old in horizons
                )

            selected = max(
                candidates,
                key=lambda candidate: (separation(candidate), -candidate),
            )
            horizons.add(selected)
        for horizon in sorted(horizons):
            target_date = origin + pd.Timedelta(days=horizon)
            if target_date > label_cutoff:
                raise ValueError("Цель выходит за конец обучающей истории.")
            rows.append(
                {
                    "origin": origin,
                    "horizon": horizon,
                    "target_date": target_date,
                    "present_in_52x2": horizon in old,
                    "present_in_52x4": horizon in four,
                    "present_in_52x8": horizon in eight,
                }
            )
    return pd.DataFrame(rows)


def clear_feature_caches():
    for function in (
        price_features_at_origin,
        regularity_features_at_origin,
        group_statistics_at_origin,
        cross_store_statistics_at_origin,
        historical_price_mean_13w,
    ):
        function.cache_clear()
    gc.collect()


def build_feature_dataset(pairs, label_cutoff=None):
    pairs = list(pairs)
    if not pairs:
        raise ValueError("Пустой план подготовки данных.")
    n_series = len(wrmsse_evaluator.series_index)
    n_rows = len(pairs) * n_series
    storage = None
    y = np.empty(n_rows, dtype=np.float32) if label_cutoff is not None else None
    if label_cutoff is not None:
        label_cutoff = pd.Timestamp(label_cutoff)
    for number, (origin, target_date) in enumerate(pairs, start=1):
        origin = pd.Timestamp(origin)
        target_date = pd.Timestamp(target_date)
        frame = make_daily_features_final(origin, target_date)
        if storage is None:
            storage = {
                column: np.empty(
                    n_rows,
                    dtype=(
                        frame[column].cat.codes.dtype
                        if column in CATEGORICAL_FEATURES
                        else frame[column].dtype
                    ),
                )
                for column in FEATURE_COLUMNS
            }
        start = (number - 1) * n_series
        stop = start + n_series
        for column in FEATURE_COLUMNS:
            values = (
                frame[column].cat.codes.to_numpy()
                if column in CATEGORICAL_FEATURES
                else frame[column].to_numpy()
            )
            storage[column][start:stop] = values
        if y is not None:
            position = daily_history_dates.get_indexer([target_date])[0]
            if (
                position < 0
                or target_date > label_cutoff
                or origin >= target_date
                or origin > label_cutoff
            ):
                raise ValueError("Недоступная дата обучающей цели.")
            y[start:stop] = daily_history_values[:, position]
            if number % (HORIZONS_PER_ORIGIN * 13) == 0:
                print(
                    f"Подготовлено дат: {number // HORIZONS_PER_ORIGIN}/52",
                    flush=True,
                )
    if y is not None and (not np.isfinite(y).all() or (y < 0).any()):
        raise ValueError("Некорректные целевые продажи.")
    data = {
        column: (
            pd.Categorical.from_codes(storage[column], dtype=CATEGORY_TYPES[column])
            if column in CATEGORICAL_FEATURES
            else storage[column]
        )
        for column in FEATURE_COLUMNS
    }
    return pd.DataFrame(data, columns=FEATURE_COLUMNS, copy=False), y


def make_window_evaluator(origin):
    history = daily_history.loc[:, :origin]
    weeks = daily_calendar.loc[history.columns[-28:], "wm_yr_wk"].unique()
    with redirect_stdout(io.StringIO()):
        evaluator = M5WRMSSE(
            history=history,
            metadata=series_metadata,
            calendar=calendar,
            prices=price_data.loc[price_data["wm_yr_wk"].isin(weeks)],
            levels=M5_LEVELS,
            horizon=HORIZON,
        )
    return evaluator


def prediction_frame(model, frame, evaluator):
    values = model.predict(frame)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Некорректные значения прогноза.")
    return pd.DataFrame(
        values.reshape(HORIZON, len(evaluator.series_index)).T,
        index=evaluator.series_index,
        columns=evaluator.forecast_dates,
    )

In [ ]:
if pd.get_option("compute.use_bottleneck"):
    raise RuntimeError("Bottleneck должен быть отключён до признаков.")
if SEARCH_ORIGIN + pd.Timedelta(days=HORIZON) > daily_history_dates[-1]:
    raise ValueError("Нет всех 28 дней validation.")
plan = make_training_plan(SEARCH_ORIGIN, HORIZONS_PER_ORIGIN)
for horizon in (1, HORIZON):
    probe = make_daily_features_final(SEARCH_ORIGIN, SEARCH_ORIGIN + pd.Timedelta(days=horizon))
    if probe.columns.tolist() != FEATURE_COLUMNS or len(probe) != 30_490:
        raise ValueError("Изменился состав признаков или число рядов.")
    if not probe.index.equals(wrmsse_evaluator.series_index):
        raise ValueError("Изменился порядок рядов.")
    if np.isinf(probe.select_dtypes(include="number").to_numpy()).any():
        raise ValueError("В признаках есть бесконечные значения.")
if plan["target_date"].max() > SEARCH_ORIGIN or plan.duplicated(["origin", "horizon"]).any():
    raise ValueError("Неверный план train.")
print("Признаков:", len(FEATURE_COLUMNS))
print("Train:", f"{len(plan) * 30_490:,}", "| Validation:", f"{HORIZON * 30_490:,}")
display(plan.tail(HORIZONS_PER_ORIGIN))
del probe
clear_feature_caches()

## 6. Отдельная история и восстановление

Этот поиск имеет новое имя `optuna_lgbm_67_cpu_52x28_v1`, отдельный SQLite и отдельный архив. Старые попытки на 52 × 8 не получают новые оценки и не переносятся в базу. Из них используются только параметры двух стартовых кандидатов.

Манифест фиксирует данные, версии, 67 признаков, план, диапазоны и правила остановки. При несовпадении протокола продолжение останавливается. Увеличить лимит попыток или время можно без подмены протокола. После аварийного завершения RUNNING помечается FAIL; её обучение не возобновляется с середины.

Checkpoint содержит завершённые результаты. SQLite не сохраняет состояние генератора TPE, поэтому следующие предложения после перезапуска могут отличаться от непрерывного запуска.

In [ ]:
def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False),
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as source:
        for block in iter(lambda: source.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def restore_checkpoint(source):
    source = Path(source)
    prefixes = [
        Path("results/m5_wrmsse") / STUDY_NAME,
        Path("models") / STUDY_NAME,
    ]
    manifest_name = prefixes[0] / "study_manifest.json"
    database_name = prefixes[1] / "study.sqlite3"
    if MANIFEST_PATH.exists() or DATABASE_PATH.exists():
        raise FileExistsError("Рабочая история уже есть; повторное восстановление не нужно.")
    if source.is_file():
        if not is_zipfile(source):
            raise ValueError("Для восстановления нужен ZIP или распакованная папка.")
        with ZipFile(source) as archive:
            files = [info for info in archive.infolist() if not info.is_dir()]
            names = {Path(info.filename) for info in files}
            if not {manifest_name, database_name}.issubset(names):
                raise ValueError("В архиве нет манифеста и SQLite именно этого поиска.")
            safe_files = []
            for info in files:
                relative = Path(info.filename)
                if relative.is_absolute() or ".." in relative.parts:
                    raise ValueError("Некорректный путь в checkpoint.")
                if not any(relative.is_relative_to(prefix) for prefix in prefixes):
                    raise ValueError(f"Лишний файл в checkpoint: {relative}")
                if (info.external_attr >> 16) & 0o170000 == 0o120000:
                    raise ValueError("Ссылки в checkpoint не поддерживаются.")
                safe_files.append((info, relative))
            for info, relative in safe_files:
                target = PROJECT_ROOT / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
    elif source.is_dir():
        if not (source / manifest_name).is_file() or not (source / database_name).is_file():
            raise ValueError("Укажи папку, внутри которой лежат results/ и models/.")
        for prefix in prefixes:
            for incoming in sorted((source / prefix).rglob("*")):
                if incoming.is_symlink():
                    raise ValueError("Ссылки в checkpoint не поддерживаются.")
                if incoming.is_file():
                    relative = incoming.relative_to(source)
                    target = PROJECT_ROOT / relative
                    target.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(incoming, target)
    else:
        raise FileNotFoundError(source)
    print("Восстановлено из:", source)


def prepare_study():
    if not MANIFEST_PATH.exists() and not DATABASE_PATH.exists():
        source = RESUME_FROM
        if source is None:
            candidates = sorted(Path("/kaggle/input").rglob(CHECKPOINT_PATH.name))
            if len(candidates) > 1:
                raise ValueError("Несколько checkpoint ZIP; укажи один в RESUME_FROM.")
            if candidates:
                source = candidates[0]
            else:
                suffix = Path("results/m5_wrmsse") / STUDY_NAME / "study_manifest.json"
                folders = sorted(
                    {
                        path.parents[3]
                        for path in Path("/kaggle/input").rglob("study_manifest.json")
                        if path.as_posix().endswith(suffix.as_posix())
                    }
                )
                if len(folders) > 1:
                    raise ValueError("Несколько распакованных историй; укажи одну в RESUME_FROM.")
                if folders:
                    source = folders[0]
        if source is not None:
            restore_checkpoint(source)
    if MANIFEST_PATH.exists() != DATABASE_PATH.exists():
        raise RuntimeError("История неполная: нужны и study_manifest.json, и study.sqlite3.")
    versions = {
        "python": platform.python_version(),
        "lightgbm": lgb.__version__,
        "optuna": optuna.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
    }
    print("Проверяю хеши трёх исходных CSV перед продолжением истории.", flush=True)
    plan = make_training_plan(SEARCH_ORIGIN, HORIZONS_PER_ORIGIN)
    raw_hashes = {name: sha256_file(DATA_RAW / name) for name in REQUIRED_FILES}
    contract = {
        "schema": 2,
        "study_name": STUDY_NAME,
        "origin": str(SEARCH_ORIGIN.date()),
        "horizon": HORIZON,
        "n_origins": N_ORIGINS,
        "horizons_per_origin": HORIZONS_PER_ORIGIN,
        "features": FEATURE_COLUMNS,
        "categorical_features": CATEGORICAL_FEATURES,
        "feature_source_sha256": FEATURE_SOURCE_SHA256,
        "raw_sha256": raw_hashes,
        "versions": versions,
        "fixed_params": FIXED_PARAMS,
        "seed_trials": SEED_TRIALS,
        "reference_results": REFERENCE_RESULTS,
        "search_space": SEARCH_SPACE,
        "max_boost_rounds": MAX_BOOST_ROUNDS,
        "early_stopping_rounds": EARLY_STOPPING_ROUNDS,
        "sampler": {"name": "TPESampler", "startup_trials": 4, "seed": SEED},
        "pruner": "NopPruner",
        "deadline_policy": "prune_current_trial_between_iterations_and_export_completed",
        "compute_use_bottleneck": False,
        "training_plan_sha256": hashlib.sha256(
            plan.to_csv(index=False, date_format="%Y-%m-%d").encode()
        ).hexdigest(),
    }
    encoded = json.dumps(contract, ensure_ascii=False, sort_keys=True).encode()
    contract_hash = hashlib.sha256(encoded).hexdigest()
    if MANIFEST_PATH.exists():
        saved = json.loads(MANIFEST_PATH.read_text())
        if saved.get("contract_sha256") != contract_hash or saved.get("contract") != contract:
            raise RuntimeError(
                "Изменились данные, версии или протокол поиска. "
                "Не смешивай результаты: восстанови прежние настройки или задай новое STUDY_NAME."
            )
    RESULT_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    storage = optuna.storages.RDBStorage(
        url=f"sqlite:///{DATABASE_PATH}",
        engine_kwargs={"connect_args": {"timeout": 60}},
    )
    study = optuna.create_study(
        study_name=STUDY_NAME,
        direction="minimize",
        storage=storage,
        load_if_exists=True,
        sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=4),
        pruner=optuna.pruners.NopPruner(),
    )
    saved_hash = study.user_attrs.get("contract_sha256")
    if saved_hash not in (None, contract_hash):
        raise RuntimeError("SQLite принадлежит другому протоколу.")
    study.set_user_attr("contract_sha256", contract_hash)
    atomic_json(MANIFEST_PATH, {"contract_sha256": contract_hash, "contract": contract})
    plan.to_csv(RESULT_DIR / "training_plan.csv", index=False, date_format="%Y-%m-%d")
    print("Версии:", versions)
    print(
        "Историческое окно:",
        SEARCH_ORIGIN.date(),
        "→",
        (SEARCH_ORIGIN + pd.Timedelta(days=28)).date(),
    )
    print(f"План: 52 × {HORIZONS_PER_ORIGIN}; строк: {len(plan) * 30_490:,}; признаков: 67")
    print("Попыток в истории:", len(study.trials))
    return study

In [ ]:
study = prepare_study()

## 7. WRMSSE и общий Dataset

Перед обучением быстрый WRMSSE сравнивается с нашей полной реализацией на одном пробном прогнозе. Порядок validation — целевой день, затем ряд. Сама метрика учитывает все 12 уровней M5.

Все попытки используют одни и те же train и validation. После построения бинарных Dataset исходная большая таблица освобождается. В логах будут подготовка 13/52, 26/52, 39/52, 52/52, размер train и итоговая проверка метрики.

In [ ]:
def build_fast_wrmsse(evaluator, actual):
    actual_values = evaluator._aligned_values(actual)
    aggregation = scipy.sparse.vstack(
        [spec["aggregation"] for spec in evaluator.level_specs.values()], format="csr"
    )
    scales = np.concatenate([spec["scale"] for spec in evaluator.level_specs.values()])
    weights = np.concatenate([spec["weights"] for spec in evaluator.level_specs.values()])
    factors = np.divide(
        weights,
        np.sqrt(scales),
        out=np.zeros_like(weights),
        where=scales > 0,
    )
    n_series, horizon = actual_values.shape

    def score(prediction):
        prediction = np.asarray(prediction, dtype=np.float64)
        if prediction.shape != (n_series * horizon,) or not np.isfinite(prediction).all():
            raise ValueError("Неверный размер или неконечный прогноз для WRMSSE.")
        errors = prediction.reshape(horizon, n_series).T - actual_values
        aggregated = aggregation @ errors
        rmse = np.sqrt(np.einsum("ij,ij->i", aggregated, aggregated) / horizon)
        return float(np.dot(factors, rmse))

    return score

In [ ]:
def build_search_data():
    started = perf_counter()
    clear_feature_caches()
    plan = make_training_plan(SEARCH_ORIGIN, HORIZONS_PER_ORIGIN)
    pairs = plan[["origin", "target_date"]].itertuples(index=False, name=None)
    X_train, y_train = build_feature_dataset(pairs, label_cutoff=SEARCH_ORIGIN)
    evaluator = make_window_evaluator(SEARCH_ORIGIN)
    valid_pairs = [(SEARCH_ORIGIN, day) for day in evaluator.forecast_dates]
    X_valid, _ = build_feature_dataset(valid_pairs)
    actual = daily_history.loc[evaluator.series_index, evaluator.forecast_dates]
    y_valid = actual.to_numpy(dtype=np.float32).T.reshape(-1)
    if len(X_train) != N_ORIGINS * HORIZONS_PER_ORIGIN * 30_490:
        raise ValueError("Неверный размер train.")
    if X_train.columns.tolist() != FEATURE_COLUMNS or len(FEATURE_COLUMNS) != 67:
        raise ValueError("Изменился состав признаков.")
    for frame in (X_train, X_valid):
        for column in CATEGORICAL_FEATURES:
            if frame[column].dtype != CATEGORY_TYPES[column] or frame[column].isna().any():
                raise ValueError(f"Неверные категории в {column}.")
    metric = build_fast_wrmsse(evaluator, actual)
    naive_values = np.repeat(
        daily_history.loc[:, :SEARCH_ORIGIN].iloc[:, -28:].mean(axis=1).to_numpy()[:, None],
        HORIZON,
        axis=1,
    )
    naive = pd.DataFrame(
        naive_values, index=evaluator.series_index, columns=evaluator.forecast_dates
    )
    ordinary, _ = evaluator.score(actual, naive)
    fast = metric(naive_values.T.reshape(-1))
    if not np.isclose(ordinary, fast, rtol=1e-12, atol=1e-12):
        raise RuntimeError("Быстрый WRMSSE отличается от M5WRMSSE.")
    print(f"Проверка быстрого WRMSSE: {fast:.8f}; совпадение: True")
    dataset_params = {
        "max_bin": FIXED_PARAMS["max_bin"],
        "bin_construct_sample_cnt": FIXED_PARAMS["bin_construct_sample_cnt"],
        "feature_pre_filter": False,
        "seed": SEED,
        "num_threads": NUM_THREADS,
        "verbosity": -1,
    }
    train_set = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=CATEGORICAL_FEATURES,
        params=dataset_params,
        free_raw_data=True,
    )
    valid_set = lgb.Dataset(
        X_valid,
        label=y_valid,
        reference=train_set,
        categorical_feature=CATEGORICAL_FEATURES,
        params=dataset_params,
        free_raw_data=True,
    )
    table_gib = float(X_train.memory_usage(index=True, deep=True).sum() / 1024**3)
    print(f"Таблица train: {table_gib:.2f} GiB. Бинирование выполняется один раз.", flush=True)
    train_set.construct()
    valid_set.construct()
    del X_train, y_train
    clear_feature_caches()
    gc.collect()
    if not np.array_equal(valid_set.get_label(), y_valid):
        raise ValueError("Порядок фактов в validation изменился.")
    atomic_json(
        RESULT_DIR / "data_preparation.json",
        {
            "train_rows": int(train_set.num_data()),
            "valid_rows": int(valid_set.num_data()),
            "features": FEATURE_COLUMNS,
            "table_gib": table_gib,
            "seconds": float(perf_counter() - started),
            "fast_wrmsse_check": {"ordinary": ordinary, "fast": fast},
            "feature_pre_filter": False,
        },
    )
    return {
        "train": train_set,
        "valid": valid_set,
        "X_valid": X_valid,
        "actual": actual,
        "evaluator": evaluator,
        "metric": metric,
    }

## 8. Что сохраняется

После каждой успешной попытки: параметры, WRMSSE, Bias, число деревьев, время, кривая обучения, важности и ошибки по уровням. Модели и полные прогнозы сохраняются для **двух лучших завершённых попыток**; метрики остальных остаются в истории.

При достижении бюджета времени сохраняется частичная кривая остановленной попытки, но её минимум не участвует в выборе финалистов. `cap_reached` означает, что достигнут лимит деревьев. Эти случаи видны отдельно от COMPLETE и ошибок.

ZIP обновляется атомарно; SQLite копируется через backup. Если Kaggle при подключении Input распаковал `predictions.csv.gz` в CSV, экспорт снова упакует CSV. На первом запуске достаточно только данных M5.

In [ ]:
def save_model_result(booster, context, name, parameters, seconds, history=None):
    trees = int(booster.best_iteration or booster.current_iteration())
    values = booster.predict(context["X_valid"], num_iteration=trees, num_threads=NUM_THREADS)
    prediction = pd.DataFrame(
        values.reshape(HORIZON, -1).T,
        index=context["evaluator"].series_index,
        columns=context["evaluator"].forecast_dates,
    )
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError("Прогноз должен быть конечным и неотрицательным.")
    metrics, levels = price_evaluate(context["evaluator"], context["actual"], prediction)
    if not np.isclose(metrics["WRMSSE"], context["metric"](values), rtol=1e-12, atol=1e-12):
        raise RuntimeError("Итоговый WRMSSE отличается от метрики ранней остановки.")
    if history is not None:
        tracked = history["valid"]["WRMSSE"][trees - 1]
        if not np.isclose(metrics["WRMSSE"], tracked, rtol=1e-10, atol=1e-10):
            raise RuntimeError("Выбранные деревья не воспроизводят минимум WRMSSE.")
    directory = RESULT_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    model_path = MODEL_DIR / f"{name}.txt"
    temporary = model_path.with_suffix(".tmp.txt")
    booster.save_model(str(temporary), num_iteration=trees)
    temporary.replace(model_path)
    prediction.to_csv(directory / "predictions.csv.gz", compression="gzip")
    levels.to_csv(directory / "metrics_by_level.csv")
    pd.Series(
        booster.feature_importance(importance_type="gain", iteration=trees),
        index=FEATURE_COLUMNS,
        name="importance_gain",
    ).sort_values(ascending=False).to_csv(directory / "feature_importance_gain.csv")
    if history is not None:
        pd.DataFrame({"WRMSSE": history["valid"]["WRMSSE"]}).rename_axis(
            "iteration_zero_based"
        ).to_csv(directory / "learning_curve.csv")
    protocol = {
        "name": name,
        "origin": str(SEARCH_ORIGIN.date()),
        "horizons_per_origin": HORIZONS_PER_ORIGIN,
        "features": FEATURE_COLUMNS,
        "params": parameters,
        "best_iteration": trees,
        "evaluated_iterations": len(history["valid"]["WRMSSE"]) if history is not None else trees,
        "fit_seconds": float(seconds),
        "metrics": metrics,
        "cap_reached": bool(
            history is not None and len(history["valid"]["WRMSSE"]) == MAX_BOOST_ROUNDS
        ),
        "historical_development_score": True,
    }
    atomic_json(directory / "protocol.json", protocol)
    return protocol

In [ ]:
def trial_table(study):
    rows = []
    for trial in study.get_trials(deepcopy=False):
        rows.append(
            {
                "trial": trial.number,
                "state": trial.state.name,
                "role": trial.user_attrs.get("role", "search"),
                "WRMSSE": trial.value if trial.state == optuna.trial.TrialState.COMPLETE else None,
                "Bias": trial.user_attrs.get("Bias"),
                "trees": trial.user_attrs.get("best_iteration"),
                "seconds": trial.user_attrs.get("fit_seconds"),
                "cap_reached": trial.user_attrs.get("cap_reached"),
                "stop_reason": trial.user_attrs.get("stop_reason"),
                **trial.params,
            }
        )
    return pd.DataFrame(rows)


def suggest_parameters(trial):
    result = {}
    for name, specification in SEARCH_SPACE.items():
        options = dict(specification)
        kind = options.pop("kind")
        if kind == "float":
            result[name] = trial.suggest_float(name, **options)
        elif kind == "categorical":
            result[name] = trial.suggest_categorical(name, **options)
        else:
            raise ValueError(f"Неизвестный тип диапазона: {kind}")
    return result


def export_checkpoint(study):
    table = trial_table(study)
    table.to_csv(RESULT_DIR / "trials.csv", index=False)
    completed = sorted(
        study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,)),
        key=lambda trial: (trial.value, trial.number),
    )
    finalists = []
    keep = {f"trial_{trial.number:03d}" for trial in completed[:2]}
    for trial in completed[:2]:
        name = f"trial_{trial.number:03d}"
        if not (MODEL_DIR / f"{name}.txt").is_file():
            raise FileNotFoundError(f"Нет модели завершённого финалиста: {name}")
        prediction_path = RESULT_DIR / name / "predictions.csv.gz"
        plain_prediction = prediction_path.with_suffix("")
        if not prediction_path.is_file() and plain_prediction.is_file():
            temporary = prediction_path.with_name("predictions.tmp.gz")
            with plain_prediction.open("rb") as incoming, gzip.open(
                temporary, "wb", compresslevel=1
            ) as outgoing:
                shutil.copyfileobj(incoming, outgoing)
            temporary.replace(prediction_path)
            plain_prediction.unlink()
        if not prediction_path.is_file():
            raise FileNotFoundError(f"Нет прогноза завершённого финалиста: {name}")
        finalists.append(
            {
                "trial": trial.number,
                "role": trial.user_attrs.get("role", "search"),
                "WRMSSE": trial.value,
                "Bias": trial.user_attrs.get("Bias"),
                "best_iteration": trial.user_attrs.get("best_iteration"),
                "params": {**FIXED_PARAMS, **trial.params},
                "horizons_per_origin": HORIZONS_PER_ORIGIN,
                "origin": str(SEARCH_ORIGIN.date()),
                "needs_independent_evaluation": True,
                "model": str((MODEL_DIR / f"{name}.txt").relative_to(PROJECT_ROOT)),
                "prediction": str(prediction_path.relative_to(PROJECT_ROOT)),
            }
        )
    atomic_json(RESULT_DIR / "finalists.json", finalists)
    for model_path in MODEL_DIR.glob("trial_*.txt"):
        if model_path.stem not in keep:
            model_path.unlink()
    for directory in RESULT_DIR.glob("trial_*"):
        if directory.is_dir() and directory.name not in keep:
            (directory / "predictions.csv.gz").unlink(missing_ok=True)
            (directory / "predictions.csv").unlink(missing_ok=True)
    atomic_json(RESULT_DIR / "reference_results.json", REFERENCE_RESULTS)
    atomic_json(
        RESULT_DIR / "run_status.json",
        {
            "completed_trials": len(completed),
            "total_trials_in_database": len(study.trials),
            "target_finished_attempts": TOTAL_TRIALS,
            "time_budget_hours_per_invocation": TIMEOUT_HOURS,
            "best_WRMSSE": completed[0].value if completed else None,
            "retained_models": sorted(keep),
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        },
    )
    temporary_archive = CHECKPOINT_PATH.with_suffix(".tmp.zip")
    with tempfile.TemporaryDirectory(prefix="optuna-db-") as temporary_directory:
        snapshot = Path(temporary_directory) / "study.sqlite3"
        with sqlite3.connect(DATABASE_PATH) as incoming, sqlite3.connect(snapshot) as outgoing:
            incoming.backup(outgoing)
        with ZipFile(temporary_archive, "w", ZIP_DEFLATED, compresslevel=1) as archive:
            for directory in (RESULT_DIR, MODEL_DIR):
                for path in sorted(directory.rglob("*")):
                    if not path.is_file() or path == DATABASE_PATH:
                        continue
                    if path.name.endswith(
                        (".tmp", ".tmp.txt", ".tmp.gz", ".lock", "-journal", "-wal", "-shm")
                    ):
                        continue
                    archive.write(path, path.relative_to(PROJECT_ROOT))
            archive.write(snapshot, DATABASE_PATH.relative_to(PROJECT_ROOT))
    temporary_archive.replace(CHECKPOINT_PATH)
    print(
        f"Checkpoint обновлён: COMPLETE {len(completed)}, архив {CHECKPOINT_PATH.name}", flush=True
    )

## 9. Поиск: до 40 попыток или 11 часов

Сначала запускаются два известных набора параметров, затем TPE. Повторного фиксированного контроля на 386 деревьях нет. Все кандидаты, включая стартовые, выбирают число деревьев одинаково — по WRMSSE на этом окне.

Таймер запускается до сборки Dataset. При исчерпании времени текущая попытка отмечается PRUNED, поиск заканчивается и экспортирует ранее завершённые результаты. После каждой итерации видны лог метрики, а каждые 100 — время попытки и оставшийся бюджет.

Повторный запуск в той же рабочей папке продолжит историю, не переобучая завершённые попытки. Одновременно запустить два поиска в одной папке не получится. Для Save & Run All нужна новая сохранённая версия, а не только открытая интерактивная вкладка.

In [ ]:
class SearchTimeBudgetExceeded(Exception):
    pass


def deadline_callback(deadline):
    def check(env):
        if perf_counter() >= deadline:
            raise SearchTimeBudgetExceeded("Исчерпан бюджет времени поиска.")

    check.order = 25
    check.before_iteration = False
    return check


def run_search(study):
    with (MODEL_DIR / "search.lock").open("w") as lock:
        try:
            fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as error:
            raise RuntimeError("Поиск уже работает; не запускай вторую копию.") from error

        interrupted = [
            trial for trial in study.trials if trial.state == optuna.trial.TrialState.RUNNING
        ]
        for trial in interrupted:
            study.tell(trial.number, state=optuna.trial.TrialState.FAIL)
        if interrupted:
            atomic_json(
                RESULT_DIR / "interrupted_trials.json",
                {
                    "marked_failed_after_resume": [trial.number for trial in interrupted],
                    "reason": "Previous process ended before study.tell completed.",
                },
            )

        finished_states = (
            optuna.trial.TrialState.COMPLETE,
            optuna.trial.TrialState.PRUNED,
            optuna.trial.TrialState.FAIL,
        )
        finished = len(study.get_trials(states=finished_states))
        if finished >= TOTAL_TRIALS:
            print("Общий бюджет попыток выполнен; обучения не будет.")
            export_checkpoint(study)
            return
        for seed in SEED_TRIALS:
            if not any(trial.user_attrs.get("role") == seed["role"] for trial in study.trials):
                study.enqueue_trial(seed["params"], user_attrs={"role": seed["role"]})

        started = perf_counter()
        deadline = started + TIMEOUT_HOURS * 3600
        context = None
        stop_reason = "search_finished"
        atomic_json(
            RESULT_DIR / "invocation_status.json",
            {
                "started_at_utc": datetime.now(timezone.utc).isoformat(),
                "budget_hours": TIMEOUT_HOURS,
                "target_finished_attempts": TOTAL_TRIALS,
                "status": "preparing_data",
            },
        )
        export_checkpoint(study)
        try:
            context = build_search_data()
            if perf_counter() >= deadline:
                stop_reason = "time_budget_during_data_preparation"
                print("Бюджет времени закончился при подготовке; новые fit не запускаем.")
                return

            def objective(trial):
                if perf_counter() >= deadline:
                    trial.set_user_attr("stop_reason", "time_budget_before_fit")
                    study.stop()
                    raise optuna.TrialPruned("Бюджет времени закончился до fit.")
                parameters = suggest_parameters(trial)
                trial.set_user_attr("role", trial.user_attrs.get("role", "search"))
                history = {}
                before = perf_counter()
                directory = RESULT_DIR / f"trial_{trial.number:03d}"
                directory.mkdir(parents=True, exist_ok=True)
                atomic_json(
                    directory / "attempt.json",
                    {
                        "trial": trial.number,
                        "role": trial.user_attrs["role"],
                        "params": {**FIXED_PARAMS, **parameters},
                        "started_at_utc": datetime.now(timezone.utc).isoformat(),
                    },
                )

                def feval(predictions, dataset):
                    if dataset is not context["valid"]:
                        raise RuntimeError("WRMSSE предназначен только для validation.")
                    return "WRMSSE", context["metric"](predictions), False

                def report(env):
                    if (env.iteration + 1) % 50 == 0:
                        value = next(
                            row[2] for row in env.evaluation_result_list if row[1] == "WRMSSE"
                        )
                        trial.report(value, step=env.iteration + 1)
                    if (env.iteration + 1) % 100 == 0:
                        minutes = (perf_counter() - before) / 60
                        remaining = max(0.0, deadline - perf_counter()) / 3600
                        print(
                            f"Trial {trial.number}: {env.iteration + 1} деревьев, {minutes:.1f} мин.; осталось {remaining:.2f} ч.",
                            flush=True,
                        )

                report.order = 22

                print(
                    f"Попытка {trial.number} ({trial.user_attrs['role']}): {parameters}", flush=True
                )
                booster = None
                try:
                    booster = lgb.train(
                        {**FIXED_PARAMS, **parameters},
                        context["train"],
                        num_boost_round=MAX_BOOST_ROUNDS,
                        valid_sets=[context["valid"]],
                        valid_names=["valid"],
                        feval=feval,
                        callbacks=[
                            lgb.log_evaluation(100),
                            lgb.record_evaluation(history),
                            report,
                            deadline_callback(deadline),
                            lgb.early_stopping(EARLY_STOPPING_ROUNDS, first_metric_only=True),
                        ],
                    )
                    protocol = save_model_result(
                        booster,
                        context,
                        f"trial_{trial.number:03d}",
                        {**FIXED_PARAMS, **parameters},
                        perf_counter() - before,
                        history,
                    )
                    for key in ("best_iteration", "fit_seconds", "cap_reached"):
                        trial.set_user_attr(key, protocol[key])
                    trial.set_user_attr("Bias", protocol["metrics"]["Bias"])
                    return protocol["metrics"]["WRMSSE"]
                except SearchTimeBudgetExceeded:
                    values = history.get("valid", {}).get("WRMSSE", [])
                    pd.DataFrame({"WRMSSE": values}).rename_axis("iteration_zero_based").to_csv(
                        directory / "partial_learning_curve.csv"
                    )
                    trial.set_user_attr("stop_reason", "time_budget")
                    trial.set_user_attr("fit_seconds", float(perf_counter() - before))
                    trial.set_user_attr("evaluated_iterations", len(values))
                    atomic_json(
                        directory / "interrupted_by_budget.json",
                        {
                            "reason": "time_budget",
                            "evaluated_iterations": len(values),
                            "fit_seconds": float(perf_counter() - before),
                            "included_in_finalists": False,
                        },
                    )
                    study.stop()
                    raise optuna.TrialPruned(
                        "11-часовой бюджет: сохранены ранее завершённые попытки."
                    )
                finally:
                    del booster
                    gc.collect()

            remaining = max(0, TOTAL_TRIALS - len(study.get_trials(states=finished_states)))
            seconds_left = max(0.0, deadline - perf_counter())
            print(
                f"Запускаем до {remaining} попыток; оставшийся бюджет {seconds_left / 3600:.2f} ч.",
                flush=True,
            )
            if remaining and seconds_left > 0:
                study.optimize(
                    objective,
                    n_trials=remaining,
                    timeout=seconds_left,
                    n_jobs=1,
                    gc_after_trial=True,
                    callbacks=[lambda current, trial: export_checkpoint(current)],
                    catch=(lgb.basic.LightGBMError, FloatingPointError),
                )
            if perf_counter() >= deadline:
                stop_reason = "time_budget"
            elif len(study.get_trials(states=finished_states)) >= TOTAL_TRIALS:
                stop_reason = "trial_budget"
        except BaseException:
            stop_reason = "exception_or_interrupt"
            raise
        finally:
            atomic_json(
                RESULT_DIR / "invocation_status.json",
                {
                    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
                    "budget_hours": TIMEOUT_HOURS,
                    "elapsed_seconds": float(perf_counter() - started),
                    "status": stop_reason,
                },
            )
            export_checkpoint(study)
            del context
            clear_feature_caches()
            gc.collect()

In [ ]:
if RUN_SEARCH:
    run_search(study)
else:
    print("Поиск выключен: RUN_SEARCH = False. Полный train Dataset не создавался.")

## 10. Итог и дальнейшая работа

Таблица показывает только завершённые оценки; прерванные по времени и ошибки видны отдельно. Сравниваем новый минимум с прежней одиночной моделью 0.501959. Отдельно показан ориентир ансамбля моделей 0.500879 — это ансамбль, а не одиночный контроль.

После поиска скачиваем этот ноутбук с выводами и `demand-planner_optuna_52x28_checkpoint.zip`. Затем используем прогнозы двух финалистов для обновления ансамбля и проверяем калибровку. Для честного сравнения с участниками нужен прогноз за период public; по нему не выбираем очередные параметры в этом поиске.

График ниже строится только по сохранённым кривым двух финалистов. Повторное обучение в итоговых ячейках не запускается.

In [ ]:
def show_search_results(study):
    table = trial_table(study)
    if table.empty:
        print("Попыток пока нет.")
    else:
        print("Состояния:", table["state"].value_counts().to_dict())
        display(
            table.sort_values("WRMSSE", na_position="last").style.format(
                {"WRMSSE": "{:.6f}", "Bias": "{:+.2%}", "seconds": "{:.0f}", "trees": "{:.0f}"},
                na_rep="—",
            )
        )
    completed = study.get_trials(states=(optuna.trial.TrialState.COMPLETE,))
    if completed:
        best = study.best_trial
        rows = [
            {"model": "Сохранённый trial 9, 52 × 28", **REFERENCE_RESULTS["single_trial9_52x28"]},
            {"model": "Сохранённый ансамбль моделей 70 / 30", **REFERENCE_RESULTS["ensemble_70_30"]},
            {
                "model": f"Новая Optuna: trial {best.number}",
                "WRMSSE": best.value,
                "Bias": best.user_attrs["Bias"],
                "trees": best.user_attrs["best_iteration"],
            },
        ]
        summary = pd.DataFrame(rows).set_index("model").drop(columns="source", errors="ignore")
        summary["gain_vs_old_single"] = (
            1 - summary["WRMSSE"] / REFERENCE_RESULTS["single_trial9_52x28"]["WRMSSE"]
        )
        summary["gain_vs_old_ensemble"] = (
            1 - summary["WRMSSE"] / REFERENCE_RESULTS["ensemble_70_30"]["WRMSSE"]
        )
        summary.to_csv(RESULT_DIR / "comparison.csv")
        display(
            summary.style.format(
                {
                    "WRMSSE": "{:.6f}",
                    "Bias": "{:+.2%}",
                    "trees": "{:.0f}",
                    "gain_vs_old_single": "{:+.2%}",
                    "gain_vs_old_ensemble": "{:+.2%}",
                },
                na_rep="—",
            )
        )
        print("Лучшие параметры:", best.params)
        print("Деревьев:", best.user_attrs["best_iteration"])
        reproduced = [
            trial for trial in completed if trial.user_attrs.get("role") == "seed_old_trial_009"
        ]
        if reproduced:
            difference = reproduced[0].value - REFERENCE_RESULTS["single_trial9_52x28"]["WRMSSE"]
            print(
                f"Повтор параметров прежнего trial 9: WRMSSE {reproduced[0].value:.6f}; разница {difference:+.6f}."
            )
        if best.user_attrs.get("cap_reached"):
            print("Лучший кандидат достиг лимита деревьев; это нужно проверить отдельно.")
        print("Окно использовалось для подбора; это не независимый Kaggle score.")
    export_checkpoint(study)
    display(FileLink(os.path.relpath(CHECKPOINT_PATH, Path.cwd())))

In [ ]:
if RUN_SEARCH or study.trials:
    show_search_results(study)
else:
    print("Поиск выключен. Для запуска оставь RUN_SEARCH = True.")

### Кривые двух финалистов

График использует только `learning_curve.csv` завершённых попыток.

In [ ]:
completed_for_plot = sorted(
    study.get_trials(states=(optuna.trial.TrialState.COMPLETE,)),
    key=lambda trial: (trial.value, trial.number),
)[:2]
if completed_for_plot:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4))
    for trial in completed_for_plot:
        curve_path = RESULT_DIR / f"trial_{trial.number:03d}" / "learning_curve.csv"
        curve = pd.read_csv(curve_path)
        ax.plot(
            np.arange(1, len(curve) + 1),
            curve["WRMSSE"],
            label=f"Trial {trial.number}: {trial.value:.6f}",
        )
        ax.scatter(trial.user_attrs["best_iteration"], trial.value, s=35)
    ax.set(xlabel="Деревья", ylabel="WRMSSE", title="52 × 28: две лучшие завершённые попытки")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(RESULT_DIR / "finalists_learning_curves.png", dpi=150)
    plt.show()
    plt.close(fig)
    export_checkpoint(study)

## Результат сохранённого поиска

За **11 часов** завершились **13 попыток**, ещё одна остановлена по времени.
Из запланированных 40 запусков начались 14. Лучшие параметры остались прежними:
trial 9 из поиска на 52 × 8 получил на плотности 28 **WRMSSE 0.501959**, **197 деревьев**.

Второй финалист — прежний trial 5 на 28: **0.503103**, **346 деревьев**.
Лучший новый набор параметров — **0.503674**; он не превзошёл контроль.
Все результаты относятся к одному историческому окну.

[Попытки](../results/m5_wrmsse/optuna_lgbm_67_cpu_52x28_v1/trials.csv).
Следующий эксперимент — недельные модели в [ноутбуке 08](08_horizon_week_models.ipynb).

### Документация

- [LightGBM 4.6.0: train и пользовательская метрика](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.train.html)
- [LightGBM 4.6.0: early stopping](https://lightgbm.readthedocs.io/en/v4.6.0/pythonapi/lightgbm.early_stopping.html)
- [LightGBM 4.6.0: параметры, включая Tweedie](https://lightgbm.readthedocs.io/en/v4.6.0/Parameters.html)
- [Optuna 4.5.0: Study, timeout и очередь попыток](https://optuna.readthedocs.io/en/v4.5.0/reference/generated/optuna.study.Study.html)
- [Optuna 4.5.0: история в SQLite](https://optuna.readthedocs.io/en/v4.5.0/tutorial/20_recipes/001_rdb.html)